In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Bawana_Delhi_DPCC_2024.xlsx")

In [4]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,360.0,214.0,287.0,150.0,272.0,308.0,97.0,159.0,86.0,145.0,372.0,303.0
1,2,353.0,260.0,127.0,182.0,246.0,162.0,131.0,90.0,90.0,195.0,333.0,304.0
2,3,350.0,238.0,137.0,166.0,322.0,165.0,114.0,66.0,108.0,179.0,405.0,301.0
3,4,371.0,305.0,138.0,192.0,303.0,207.0,57.0,69.0,75.0,188.0,412.0,189.0
4,5,316.0,238.0,128.0,198.0,331.0,312.0,72.0,NaN,66.0,127.0,414.0,186.0
5,6,319.0,134.0,163.0,201.0,286.0,172.0,61.0,67.0,134.0,129.0,401.0,235.0
6,7,344.0,206.0,216.0,252.0,327.0,210.0,57.0,51.0,77.0,116.0,427.0,278.0
7,8,355.0,214.0,199.0,237.0,292.0,245.0,64.0,66.0,82.0,169.0,425.0,362.0
8,9,297.0,236.0,166.0,205.0,213.0,186.0,73.0,60.0,114.0,187.0,404.0,215.0
9,10,299.0,338.0,209.0,276.0,242.0,168.0,177.0,74.0,100.0,145.0,390.0,278.0


In [3]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,360.0,214.000000,287.0,150.000000,272.0,170.228571,97.000000,69.757576,86.000000,145.0,372.000000,303.0
1,2,353.0,260.000000,127.0,182.000000,246.0,162.000000,131.000000,90.000000,90.000000,195.0,333.000000,304.0
2,3,350.0,238.000000,137.0,166.000000,322.0,165.000000,114.000000,66.000000,108.000000,179.0,405.000000,301.0
3,4,371.0,305.000000,138.0,192.000000,303.0,207.000000,57.000000,69.000000,75.000000,188.0,412.000000,189.0
4,5,316.0,238.000000,128.0,198.000000,331.0,170.228571,72.000000,69.757576,66.000000,127.0,414.000000,186.0
5,6,319.0,134.000000,163.0,201.000000,286.0,172.000000,61.000000,67.000000,134.000000,129.0,401.000000,235.0
6,7,344.0,206.000000,216.0,252.000000,327.0,210.000000,57.000000,51.000000,77.000000,116.0,427.000000,278.0
7,8,355.0,214.000000,199.0,237.000000,292.0,245.000000,64.000000,66.000000,82.000000,169.0,425.000000,362.0
8,9,297.0,236.000000,166.0,205.000000,213.0,186.000000,73.000000,60.000000,114.000000,187.0,404.000000,215.0
9,10,299.0,338.000000,209.0,276.000000,242.0,168.000000,88.882353,74.000000,100.000000,145.0,390.000000,278.0
